# 01 — Data Preparation

Download, merge, and preprocess three TCGA Pan-Cancer Atlas datasets for TMB prediction.

**Datasets:**
1. TCGA-CDR clinical data (Liu et al. 2018)
2. Mutation counts + genomic features from cBioPortal
3. Aneuploidy/WGD data (Taylor et al. 2018)

**Outputs:** `data/processed/tmb_merged.parquet` and `data/processed/tmb_merged.csv`

In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import (
    download_tcga_cdr, load_tcga_cdr,
    download_cbioportal_clinical, load_cbioportal_clinical,
    download_aneuploidy_data, load_aneuploidy_data,
)
from src.preprocessing import merge_datasets, compute_tmb, clean_and_encode

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/leojo/Developer/math189/tmb-prediction_math189


## 1. Download Datasets

In [2]:
print("=== Dataset 1: TCGA-CDR Clinical ===")
download_tcga_cdr(RAW_DIR)

print("\n=== Dataset 2: cBioPortal Clinical ===")
download_cbioportal_clinical(RAW_DIR)

print("\n=== Dataset 3: Aneuploidy / WGD ===")
download_aneuploidy_data(RAW_DIR)

print("\nAll downloads complete.")

=== Dataset 1: TCGA-CDR Clinical ===
  [cached] TCGA-CDR.xlsx

=== Dataset 2: cBioPortal Clinical ===
  [cached] cbioportal_clinical.tsv

=== Dataset 3: Aneuploidy / WGD ===
  [cached] taylor_aneuploidy.tsv
  [cached] taylor_absolute_purity.tsv

All downloads complete.


## 2. Load and Inspect Each Dataset

In [3]:
# Dataset 1: TCGA-CDR
cdr_df = load_tcga_cdr(RAW_DIR)
print(f"TCGA-CDR shape: {cdr_df.shape}")
print(f"Columns: {list(cdr_df.columns[:20])}")
print(f"\nMissing values (top 10):")
print(cdr_df.isnull().sum().sort_values(ascending=False).head(10))
cdr_df.head(3)

TCGA-CDR shape: (11160, 34)
Columns: ['Unnamed: 0', 'patient_barcode', 'type', 'age_at_initial_pathologic_diagnosis', 'gender', 'race', 'ajcc_pathologic_tumor_stage', 'clinical_stage', 'histological_type', 'histological_grade', 'initial_pathologic_dx_year', 'menopause_status', 'birth_days_to', 'vital_status', 'tumor_status', 'last_contact_days_to', 'death_days_to', 'cause_of_death', 'new_tumor_event_type', 'new_tumor_event_site']

Missing values (top 10):
Redaction                     11103
residual_tumor                 9883
margin_status                  9802
new_tumor_event_site_other     9725
new_tumor_event_site           8893
new_tumor_event_type           8410
new_tumor_event_dx_days_to     7814
death_days_to                  7563
DFI.time                       5639
DFI                            5629
dtype: int64


,Unnamed: 0,patient_barcode,type,age_at_initial_pathologic_diagnosis,gender,race,ajcc_pathologic_tumor_stage,clinical_stage,histological_type,histological_grade,...,residual_tumor,OS,OS.time,DSS,DSS.time,DFI,DFI.time,PFI,PFI.time,Redaction
0,1,TCGA-OR-A5J1,ACC,58.0,MALE,WHITE,Stage II,[Not Applicable],Adrenocortical carcinoma- Usual Type,[Not Available],...,NaN,1.0,1355.0,1.0,1355.0,1.0,754.0,1.0,754.0,NaN
1,2,TCGA-OR-A5J2,ACC,44.0,FEMALE,WHITE,Stage IV,[Not Applicable],Adrenocortical carcinoma- Usual Type,[Not Available],...,NaN,1.0,1677.0,1.0,1677.0,NaN,NaN,1.0,289.0,NaN
2,3,TCGA-OR-A5J3,ACC,23.0,FEMALE,WHITE,Stage III,[Not Applicable],Adrenocortical carcinoma- Usual Type,[Not Available],...,NaN,0.0,2091.0,0.0,2091.0,1.0,53.0,1.0,53.0,NaN


In [4]:
# Dataset 2: cBioPortal
cbio_df = load_cbioportal_clinical(RAW_DIR)
print(f"cBioPortal shape: {cbio_df.shape}")
print(f"Columns: {list(cbio_df.columns)}")
print(f"\nMissing values:")
print(cbio_df.isnull().sum())
cbio_df.head(3)

cBioPortal shape: (10967, 10)
Columns: ['sample_id', 'patient_id', 'study_id', 'aneuploidy_score', 'cancer_type', 'cancer_type_detailed', 'fraction_genome_altered', 'msi_score_mantis', 'msi_sensor_score', 'mutation_count']

Missing values:
sample_id                     0
patient_id                    0
study_id                      0
aneuploidy_score            527
cancer_type                   0
cancer_type_detailed          0
fraction_genome_altered     200
msi_score_mantis           1255
msi_sensor_score            913
mutation_count              863
dtype: int64


,sample_id,patient_id,study_id,aneuploidy_score,cancer_type,cancer_type_detailed,fraction_genome_altered,msi_score_mantis,msi_sensor_score,mutation_count
0,TCGA-02-0001-01,TCGA-02-0001,gbm_tcga_pan_can_atlas_2018,15.0,Glioblastoma,Glioblastoma Multiforme,0.2459,NaN,NaN,NaN
1,TCGA-02-0003-01,TCGA-02-0003,gbm_tcga_pan_can_atlas_2018,4.0,Glioblastoma,Glioblastoma Multiforme,0.1480,0.273,0.07,44.0
2,TCGA-02-0006-01,TCGA-02-0006,gbm_tcga_pan_can_atlas_2018,11.0,Glioblastoma,Glioblastoma Multiforme,0.2391,NaN,NaN,NaN


In [5]:
# Dataset 3: Aneuploidy / WGD
aneuploidy_df = load_aneuploidy_data(RAW_DIR)
print(f"Aneuploidy data shape: {aneuploidy_df.shape}")
print(f"Columns: {list(aneuploidy_df.columns[:15])}")
print(f"\nMissing values (top 10):")
print(aneuploidy_df.isnull().sum().sort_values(ascending=False).head(10))
aneuploidy_df.head(3)

Aneuploidy data shape: (11587, 53)
Columns: ['Sample', 'Type', 'Aneuploidy Score', '1p', '1q', '2p', '2q', '3p', '3q', '4p', '4q', '5p', '5q', '6p', '6q']

Missing values (top 10):
1p          2771
17q         2736
13 (13q)    2590
11q         2571
19q         2482
2q          2445
19p         2435
3q          2397
8q          2374
15 (15q)    2367
dtype: int64


,Sample,Type,Aneuploidy Score,1p,1q,2p,2q,3p,3q,4p,...,array,sample,call status,purity,ploidy,Genome doublings,Coverage for 80% power,Cancer DNA fraction,Subclonal genome fraction,solution
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,TCGA-P6-A5OG-01,ACC-TCGA-P6-A5OG-Tumor,maf_call,0.38,1.48,0.0,19.0,0.32,NaN,old
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,TCGA-KL-8327-01,BCM-KICH-TCGA-KL-8327-01A-11D-2310-10,maf_call,0.84,3.04,1.0,14.0,0.89,NaN,old
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,TCGA-KL-8333-01,BCM-KICH-TCGA-KL-8333-01A-11D-2310-10,maf_call,0.41,1.76,0.0,19.0,0.38,NaN,old


## 3. Merge Datasets

In [6]:
merged = merge_datasets(cdr_df, cbio_df, aneuploidy_df)
print(f"Merged shape: {merged.shape}")
print(f"\nSample flow:")
print(f"  cBioPortal samples:  {cbio_df.shape[0]}")
print(f"  TCGA-CDR patients:   {cdr_df.shape[0]}")
print(f"  Aneuploidy patients: {aneuploidy_df['patient_barcode'].nunique()}")
print(f"  After merge:         {merged.shape[0]}")
print(f"  Unique patients:     {merged['patient_barcode'].nunique()}")

Merged shape: (10953, 96)

Sample flow:
  cBioPortal samples:  10967
  TCGA-CDR patients:   11160
  Aneuploidy patients: 10757
  After merge:         10953
  Unique patients:     10953


## 4. Compute TMB

In [7]:
merged = compute_tmb(merged)

print("TMB summary:")
print(merged[["tmb", "log_tmb"]].describe())
print(f"\nTMB-high prevalence: {merged['tmb_high'].mean():.1%} ({merged['tmb_high'].sum()} / {merged['tmb_high'].notna().sum()})")

TMB summary:
                tmb       log_tmb
count  10090.000000  10090.000000
mean       7.062841      1.316275
std       27.170401      0.939325
min        0.033333      0.032790
25%        0.933333      0.659246
50%        1.966667      1.087439
75%        4.466667      1.698669
max      856.566667      6.754099

TMB-high prevalence: 11.5% (1255 / 10953)


## 5. Clean and Encode Features

In [8]:
df = clean_and_encode(merged)

print(f"Final shape: {df.shape}")
print(f"\nKey column dtypes:")
key_cols = [
    "patient_barcode", "cancer_type", "age_at_diagnosis", "sex",
    "mutation_count", "tmb", "log_tmb", "tmb_high",
    "fraction_genome_altered", "aneuploidy_score",
    "msi_status", "wgd_status",
]
for col in key_cols:
    if col in df.columns:
        n_valid = df[col].notna().sum()
        print(f"  {col:30s} {str(df[col].dtype):10s} {n_valid:>6d} non-null")
    else:
        print(f"  {col:30s} ** NOT FOUND **")

Final shape: (10953, 103)

Key column dtypes:
  patient_barcode                object      10953 non-null
  cancer_type                    object      10953 non-null
  age_at_diagnosis               float64     10815 non-null
  sex                            object      10953 non-null
  mutation_count                 float64     10090 non-null
  tmb                            float64     10090 non-null
  log_tmb                        float64     10090 non-null
  tmb_high                       int64       10953 non-null
  fraction_genome_altered        float64     10753 non-null
  aneuploidy_score               float64     10430 non-null
  msi_status                     object      10953 non-null
  wgd_status                     int64       10953 non-null


In [9]:
# Missingness summary
print("Missing values for key analysis variables:")
for col in key_cols:
    if col in df.columns:
        n_miss = df[col].isna().sum()
        pct = n_miss / len(df) * 100
        print(f"  {col:30s} {n_miss:>5d} ({pct:.1f}%)")

Missing values for key analysis variables:
  patient_barcode                    0 (0.0%)
  cancer_type                        0 (0.0%)
  age_at_diagnosis                 138 (1.3%)
  sex                                0 (0.0%)
  mutation_count                   863 (7.9%)
  tmb                              863 (7.9%)
  log_tmb                          863 (7.9%)
  tmb_high                           0 (0.0%)
  fraction_genome_altered          200 (1.8%)
  aneuploidy_score                 523 (4.8%)
  msi_status                         0 (0.0%)
  wgd_status                         0 (0.0%)


## 6. Save Processed Data

In [10]:
# Save as parquet (efficient) and CSV (human-readable)
parquet_path = PROCESSED_DIR / "tmb_merged.parquet"
csv_path = PROCESSED_DIR / "tmb_merged.csv"

df.to_parquet(parquet_path, index=False)
df.to_csv(csv_path, index=False)

print(f"Saved: {parquet_path} ({parquet_path.stat().st_size / 1e6:.1f} MB)")
print(f"Saved: {csv_path} ({csv_path.stat().st_size / 1e6:.1f} MB)")

Saved: /Users/leojo/Developer/math189/tmb-prediction_math189/data/processed/tmb_merged.parquet (1.3 MB)
Saved: /Users/leojo/Developer/math189/tmb-prediction_math189/data/processed/tmb_merged.csv (7.6 MB)


## 7. Data Dictionary

In [11]:
data_dict = {
    "Variable": [
        "patient_barcode", "cancer_type", "age_at_diagnosis", "sex",
        "mutation_count", "tmb", "log_tmb", "tmb_high",
        "fraction_genome_altered", "aneuploidy_score",
        "msi_status", "msi_score_mantis", "wgd_status",
    ],
    "Type": [
        "ID", "categorical", "continuous", "categorical",
        "count", "continuous", "continuous", "binary",
        "continuous", "continuous",
        "categorical", "continuous", "binary",
    ],
    "Description": [
        "12-char TCGA patient barcode",
        "Cancer type abbreviation (33 types)",
        "Age at initial diagnosis (years)",
        "Patient sex (MALE/FEMALE)",
        "Total somatic mutation count",
        "Tumor Mutational Burden (mut/Mb) = mutation_count / 30",
        "Log-transformed TMB: log(1 + TMB)",
        "TMB-high indicator (1 if TMB >= 10 mut/Mb)",
        "Fraction of genome with copy number alteration",
        "Aneuploidy score (arm-level events)",
        "MSI status: MSI-H (MANTIS >= 0.4) or MSS",
        "MANTIS MSI score (continuous)",
        "Whole-genome doubling status (0/1)",
    ],
}

pd.DataFrame(data_dict).style.set_properties(**{"text-align": "left"})

,Variable,Type,Description
0,patient_barcode,ID,12-char TCGA patient barcode
1,cancer_type,categorical,Cancer type abbreviation (33 types)
2,age_at_diagnosis,continuous,Age at initial diagnosis (years)
3,sex,categorical,Patient sex (MALE/FEMALE)
4,mutation_count,count,Total somatic mutation count
5,tmb,continuous,Tumor Mutational Burden (mut/Mb) = mutation_count / 30
6,log_tmb,continuous,Log-transformed TMB: log(1 + TMB)
7,tmb_high,binary,TMB-high indicator (1 if TMB >= 10 mut/Mb)
8,fraction_genome_altered,continuous,Fraction of genome with copy number alteration
9,aneuploidy_score,continuous,Aneuploidy score (arm-level events)
